# Notebook 06 · La capa `Embedding` de Keras

> **Unidad 2 · NLP e IA Generativa — Clase 1: Text Mining y Embeddings**  
> Ruta de la clase: 01 Texto → 02 Limpieza → 03 Tokenización → 04 Padding → 05 BoW/TF-IDF → **06 Embeddings** → 07 Clasificador → 08 Laboratorio

## Objetivo
Entender la capa `Embedding` como una **tabla de búsqueda entrenable** $W \in \mathbb{R}^{V \times d}$, interpretar sus parámetros y *shapes*, y **ver** cómo se organizan los vectores después de entrenar.

- $V$ = `input_dim` (tamaño del vocabulario + 1 por el padding)
- $d$ = `output_dim` (dimensión de cada vector)

```
índices (batch, maxlen)  ──►  Embedding  ──►  vectores (batch, maxlen, d)
```

⏱️ Tiempo sugerido: 20 min

> 🧑‍🎓 **Versión para estudiantes.** Las celdas de código están vacías: complétalas en clase siguiendo las explicaciones.
> Cada celda trae **pistas** (funciones útiles) y los **nombres de variables** que deben quedar definidos, porque las celdas siguientes los usan.
> Ejecuta las celdas en orden. La versión resuelta está en la carpeta `notebooks/`.

In [ ]:
# ✅ Celda de configuración (ya resuelta): ejecútala antes de empezar.

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
from keras import layers
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

keras.utils.set_random_seed(42)   # reproducibilidad

## 1) Una Embedding es una tabla de búsqueda
Antes de entrenar, los vectores son **aleatorios**. Pedir el embedding del índice 3 es literalmente leer la **fila 3** de la matriz de pesos.

In [ ]:
# ✍️ Ejercicio 1: escribe aquí tu código siguiendo la explicación de arriba.
# 💡 Pistas (funciones/métodos útiles): Embedding, array, get_weights, allclose
# 📌 Al terminar deberían existir: emb, salida, W


## 2) Datos: opiniones de clientes
Mismo pipeline de los notebooks anteriores: limpieza → tokenización → padding.

In [ ]:
# ✍️ Ejercicio 2: escribe aquí tu código siguiendo la explicación de arriba.
# 💡 Pistas (funciones/métodos útiles): read_csv, apply, Tokenizer, fit_on_texts, pad_sequences, lower, sub, strip
# 📌 Al terminar deberían existir: limpiar_texto(), df, df['limpio'], y, tokenizer, MAX_LEN, X, VOCAB


## 3) Modelo mínimo
`Embedding → GlobalAveragePooling1D → Dense(ReLU) → Dense(sigmoid)`

- `keras.Input(shape=(MAX_LEN,))` define la longitud de entrada (en Keras 3 reemplaza al antiguo `input_length`, que está obsoleto).
- `GlobalAveragePooling1D` promedia los vectores de la frase → un único vector por texto.
- Usamos `output_dim=8` para poder inspeccionar los vectores con facilidad.

In [ ]:
# ✍️ Ejercicio 3: escribe aquí tu código siguiendo la explicación de arriba.
# 💡 Pistas (funciones/métodos útiles): Sequential, compile, summary, Input, Embedding, GlobalAveragePooling1D, Dense, Adam
# 📌 Al terminar deberían existir: EMB_DIM, model


## 4) Shapes paso a paso

In [ ]:
# ✍️ Ejercicio 4: escribe aquí tu código siguiendo la explicación de arriba.
# 💡 Pistas (funciones/métodos útiles): get_layer, model
# 📌 Al terminar deberían existir: muestra, capa_emb


## 5) Entrenar y comparar vectores antes / después
Guardamos los pesos antes de entrenar para ver cómo **el entrenamiento organiza el espacio**.

In [ ]:
# ✍️ Ejercicio 5: escribe aquí tu código siguiendo la explicación de arriba.
# 💡 Pistas (funciones/métodos útiles): copy, fit, get_weights
# 📌 Al terminar deberían existir: W_antes, history, W_despues


In [ ]:
# ✍️ Ejercicio 6: escribe aquí tu código siguiendo la explicación de arriba.
# 💡 Pistas (funciones/métodos útiles): DataFrame, append, cosine_similarity
# 📌 Al terminar deberían existir: vec(), pares, filas


> 🔎 Después de entrenar, palabras con el **mismo sentimiento** tienden a coseno positivo y las opuestas a coseno negativo. El modelo aprendió una "dirección de sentimiento" en el espacio, **solo a partir de las etiquetas**.

## 6) Visualizar el espacio de embeddings (PCA a 2D)

In [ ]:
# ✍️ Ejercicio 7: escribe aquí tu código siguiendo la explicación de arriba.
# 💡 Pistas (funciones/métodos útiles): subplots, suptitle, tight_layout, show, fit_transform, scatter, set_title, grid
# 📌 Al terminar deberían existir: palabras, fig, axes


## 7) Predicción sobre texto nuevo
Siempre el **mismo** pipeline: `limpiar_texto` → `tokenizer` → `pad_sequences`.

In [ ]:
# ✍️ Ejercicio 8: escribe aquí tu código siguiendo la explicación de arriba.
# 💡 Pistas (funciones/métodos útiles): pad_sequences, texts_to_sequences, limpiar_texto, predict


## Para llevar
- La Embedding **no sabe nada** al inicio: aprende de la tarea (aquí, sentimiento).
- Con más datos y tareas más generales (Word2Vec, GloVe, FastText) los vectores capturan relaciones semánticas ricas.
- Limitación: **un vector por palabra**, sin importar el contexto (`banco` de dinero vs. `banco` del parque) → lo resuelven los **Transformers** (próxima clase).

## Preguntas de reflexión
1. ¿Qué efecto tendría duplicar `output_dim`? ¿Y en el número de parámetros?
2. ¿Qué sucede si `input_dim` es menor que el índice más grande del vocabulario?
3. ¿Por qué la salida de Embedding tiene 3 dimensiones?
4. ¿Qué hace `mask_zero=True` y por qué importa con `GlobalAveragePooling1D`?